# Project FORESIGHT — EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

DATA_DIR = "../data"
sales = pd.read_pickle(os.path.join(DATA_DIR, "sales_complete.pkl"))
inventory = pd.read_pickle(os.path.join(DATA_DIR, "inventory_clean.pkl"))

print("Sales:", sales.shape)
print("Inventory:", inventory.shape)
display(sales.head())

In [ ]:
# Missing values
display(sales.isna().sum().sort_values(ascending=False).to_frame("missing"))

In [ ]:
# Daily sales trend
daily = sales.groupby("date", as_index=False)["units_sold"].sum()
plt.figure(figsize=(14,5))
plt.plot(daily["date"], daily["units_sold"])
plt.title("Daily Units Sold")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Category sales
category = sales.groupby("category")["units_sold"].sum().sort_values(ascending=False)
display(category.to_frame())
plt.figure(figsize=(10,5))
category.plot(kind="bar")
plt.title("Sales by Category")
plt.ylabel("Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Promotion impact
display(sales.groupby("promotion")["units_sold"].agg(["count","mean","sum"]))

In [ ]:
# Seasonal analysis
display(sales.groupby("season")["units_sold"].agg(["count","mean","sum"]))

In [ ]:
# Top 10 SKUs
top10 = sales.groupby("sku_id")["units_sold"].sum().nlargest(10)
display(top10.to_frame())
plt.figure(figsize=(10,5))
top10.plot(kind="bar")
plt.title("Top 10 SKUs")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()

In [ ]:
# Create weekly dataset for forecasting
sales = sales.sort_values(["sku_id","date"])
sales["week_start"] = sales["date"] - pd.to_timedelta(sales["date"].dt.dayofweek, unit="D")
weekly = sales.groupby(["sku_id","week_start"], as_index=False).agg(
    units_sold=("units_sold","sum"),
    revenue=("revenue","sum"),
    avg_price=("unit_price_sales","mean"),
    promotion=("promotion","max"),
    holiday_flag=("holiday_flag","max")
)
weekly.to_pickle(os.path.join(DATA_DIR,"weekly_sales.pkl"))
print("Weekly data saved:", weekly.shape)